# 01 · Build source-space graphs (pipeline v3)

**How to use:** *Runtime → Run all*. Nothing to edit. Allow Google Drive access when asked.

If Colab disconnects or the runtime ends, open the notebook again and *Run all* again: finished
work is kept on Google Drive and skipped, and the computation continues where it stopped.
Everything (checkpoints, results, figures, manifests) is written to Drive, never to the
temporary Colab disk.

Steps (each stops the notebook with a clear message if something is wrong):
1. setup and input check: 20 + 23 recordings, labels 10 AD / 10 HC and 10 AD / 13 HC;
2. synthetic test suite (once per code version, about 10–15 min);
3. EDF header check (all 30 channels used by the pipeline present);
4. smoke test on the first subject of each dataset (downloads fsaverage into the Drive cache once);
5. full graph construction for all 43 recordings (several hours; resumable);
6. quality summary.

Then open one experiment notebook in `notebooks/experiments/` (e.g. `Exp0_sensor_psd.ipynb`).
A GPU is not needed for this notebook.

In [ ]:
# ===== Setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_CODE = False   # leave False. True only if you were asked to switch to newer code
                      # (all notebooks then use the newest commit; unfinished work is recomputed).
BRANCH = 'claude/ecstatic-davinci-a2e8mh'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py'] + (['--update-code'] if UPDATE_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
print('outputs on Drive:', OUT)

## 1. Input check (file names and participant sheets only)

In [ ]:
from adgraph.colab import check_inputs
check_inputs(PATHS)

## 2. Synthetic test suite (no EEG)
Must pass with zero dependency-related skips. Skipped automatically if it already passed for this code.

In [ ]:
from adgraph.colab import run_tests_once
run_tests_once(OUT)

## 3. EDF header check (reads no signal)

In [ ]:
sh(f"python scripts/inspect_raw.py --paths configs/paths.yaml --out '{OUT}' --strict")

## 4. Smoke test: first subject of each dataset
Written to Drive as `graphs_v3_<hash>_limit1/`; it is never used for analysis.

In [ ]:
sh("python scripts/build_graphs.py --paths configs/paths.yaml --limit 1")

## 5. Full graph construction (resumable)
Progress is saved per subject. If the runtime stops, *Run all* again.

In [ ]:
sh("python scripts/build_graphs.py --paths configs/paths.yaml --status")
sh("python scripts/build_graphs.py --paths configs/paths.yaml")

## 6. Quality summary

In [ ]:
import json
import pandas as pd
from adgraph.experiments.gate import gate_failures
from adgraph.graphs.store import GraphStore
from adgraph.pipeline import latest_complete_graph_run

run = latest_complete_graph_run(OUT)
m = json.load(open(run / 'manifest.json'))
print('graph run:', run)
print('commit', m['git_commit'], '| config', m['config_hash'], '| graph_run_id', m['graph_run_id'],
      '| operator error', m['source_operator']['verification_rel_error'])
print(json.dumps(m['summary']['per_dataset'], indent=1))
print('subjects not ok:', m['summary']['subjects_not_ok'] or 'none')
fails = gate_failures(GraphStore.load(run, keys=()))
print('cross-dataset gate (DESIGN.md §7):', 'PASSED' if not fails else 'NOT passed: ' + '; '.join(fails))
s = pd.read_csv(run / 'subjects.csv')
s[['key', 'label', 'orig_sfreq', 'duration_s', 'median_std_uv', 'n_bad_channels', 'n_windows_kept',
   'n_windows_total', 'status']]